In [23]:
import pandas as pd
import json
import statistics
from pydantic import BaseModel
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda

from dotenv import load_dotenv
load_dotenv()

True

# Ingestion & Analyse de données techniques

In [24]:
dataset = pd.read_json("rapport.json")

In [25]:
dataset.describe()

,cpu_usage,memory_usage,latency_ms,disk_usage,network_in_kbps,network_out_kbps,io_wait,thread_count,active_connections,error_rate,uptime_seconds,temperature_celsius,power_consumption_watts
count,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000,5.000000e+02,500.000000,500.000000
mean,60.668000,67.628000,156.018000,63.906000,1361.508000,1219.774000,4.010000,151.504000,58.862000,0.031580,8.091000e+05,63.002000,263.982000
std,12.276122,7.389101,66.994733,9.974776,431.733132,316.454341,2.717475,7.270744,23.734228,0.030291,2.600673e+05,7.725112,36.564403
min,52.000000,62.000000,113.000000,57.000000,1080.000000,991.000000,2.000000,143.000000,48.000000,0.020000,3.600000e+05,57.000000,238.000000
25%,54.000000,63.000000,124.000000,59.000000,1147.750000,1050.000000,3.000000,146.000000,49.000000,0.020000,5.845500e+05,59.000000,246.000000
50%,56.000000,66.000000,132.000000,60.000000,1221.000000,1118.500000,3.000000,151.000000,50.000000,0.020000,8.091000e+05,61.000000,253.000000
75%,59.000000,67.000000,139.000000,62.000000,1298.000000,1185.250000,3.000000,155.000000,52.000000,0.020000,1.033650e+06,62.000000,259.000000
max,99.000000,92.000000,384.000000,97.000000,2854.000000,2301.000000,14.000000,189.000000,136.000000,0.130000,1.258200e+06,89.000000,388.000000


In [26]:
dataset.columns

Index(['timestamp', 'cpu_usage', 'memory_usage', 'latency_ms', 'disk_usage',
       'network_in_kbps', 'network_out_kbps', 'io_wait', 'thread_count',
       'active_connections', 'error_rate', 'uptime_seconds',
       'temperature_celsius', 'power_consumption_watts', 'service_status'],
      dtype='str')

In [27]:
cols = ['cpu_usage','memory_usage','latency_ms','disk_usage','network_in_kbps','network_out_kbps','io_wait','thread_count','active_connections','error_rate','temperature_celsius','power_consumption_watts']
for c in cols:
    vals = dataset[c]
    print(f'{c}: min={min(vals)} max={max(vals)} mean={statistics.mean(vals):.2f} stdev={statistics.stdev(vals):.2f}')

cpu_usage: min=52 max=99 mean=60.67 stdev=12.28
memory_usage: min=62 max=92 mean=67.63 stdev=7.39
latency_ms: min=113 max=384 mean=156.02 stdev=66.99
disk_usage: min=57 max=97 mean=63.91 stdev=9.97
network_in_kbps: min=1080 max=2854 mean=1361.51 stdev=431.73
network_out_kbps: min=991 max=2301 mean=1219.77 stdev=316.45
io_wait: min=2 max=14 mean=4.01 stdev=2.72
thread_count: min=143 max=189 mean=151.50 stdev=7.27
active_connections: min=48 max=136 mean=58.86 stdev=23.73
error_rate: min=0.02 max=0.13 mean=0.03 stdev=0.03
temperature_celsius: min=57 max=89 mean=63.00 stdev=7.73
power_consumption_watts: min=238 max=388 mean=263.98 stdev=36.56


In [28]:
schemas = []
for status in dataset['service_status'].to_list():
    if status not in schemas:
        schemas.append(status)

In [29]:
schemas

[{'database': 'online', 'api_gateway': 'degraded', 'cache': 'online'},
 {'database': 'online', 'api_gateway': 'online', 'cache': 'online'},
 {'database': 'online', 'api_gateway': 'degraded', 'cache': 'degraded'},
 {'database': 'offline', 'api_gateway': 'online', 'cache': 'online'},
 {'database': 'online', 'api_gateway': 'online', 'cache': 'degraded'}]

# Detecteur d'anomalie

Les valeurs du service_status sont un ensemble de booléen. On cherche à créer un détecteur d'anomalie donc on défini:
- Error Status: True
- Normal Status: False

In [30]:
SERVICES = ["database", "api_gateway", "cache"]

raw_status_df = dataset["service_status"].apply(pd.Series)

status_df = raw_status_df.copy()
for col in SERVICES:
    status_df[col] = status_df[col] != "online"

In [31]:
status_df.describe()

,database,api_gateway,cache
count,500,500,500
unique,2,2,2
top,False,False,False
freq,441,433,466


In [32]:
dataset = pd.concat([dataset, status_df], axis=1)
dataset.head()

,timestamp,cpu_usage,memory_usage,latency_ms,disk_usage,network_in_kbps,network_out_kbps,io_wait,thread_count,active_connections,error_rate,uptime_seconds,temperature_celsius,power_consumption_watts,service_status,database,api_gateway,cache
0,2023-10-01 12:00:00+00:00,93,86,334,89,2541,2137,12,143,126,0.12,360000,84,356,"{'database': 'online', 'api_gateway': 'degrade...",False,True,False
1,2023-10-01 12:30:00+00:00,57,66,139,61,1171,1193,3,145,51,0.02,361800,58,253,"{'database': 'online', 'api_gateway': 'online'...",False,False,False
2,2023-10-01 13:00:00+00:00,56,68,136,62,1316,1147,3,147,49,0.02,363600,59,242,"{'database': 'online', 'api_gateway': 'online'...",False,False,False
3,2023-10-01 13:30:00+00:00,53,62,142,58,1090,1121,3,154,51,0.02,365400,62,240,"{'database': 'online', 'api_gateway': 'online'...",False,False,False
4,2023-10-01 14:00:00+00:00,52,67,123,61,1086,1123,3,145,48,0.02,367200,59,249,"{'database': 'online', 'api_gateway': 'online'...",False,False,False


In [33]:
dataset[dataset.api_gateway == True]

,timestamp,cpu_usage,memory_usage,latency_ms,disk_usage,network_in_kbps,network_out_kbps,io_wait,thread_count,active_connections,error_rate,uptime_seconds,temperature_celsius,power_consumption_watts,service_status,database,api_gateway,cache
0,2023-10-01 12:00:00+00:00,93,86,334,89,2541,2137,12,143,126,0.12,360000,84,356,"{'database': 'online', 'api_gateway': 'degrade...",False,True,False
6,2023-10-01 15:00:00+00:00,73,79,213,77,1506,1618,6,165,73,0.04,370800,67,290,"{'database': 'online', 'api_gateway': 'degrade...",False,True,True
7,2023-10-01 15:30:00+00:00,72,76,232,73,1455,1391,5,157,69,0.04,372600,71,272,"{'database': 'online', 'api_gateway': 'degrade...",False,True,True
8,2023-10-01 16:00:00+00:00,73,75,225,76,1651,1486,6,156,70,0.04,374400,70,291,"{'database': 'online', 'api_gateway': 'degrade...",False,True,True
9,2023-10-01 16:30:00+00:00,75,75,200,76,1645,1529,6,163,72,0.04,376200,73,277,"{'database': 'online', 'api_gateway': 'degrade...",False,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
450,2023-10-10 21:00:00+00:00,98,92,315,95,2538,2256,11,154,128,0.11,1170000,86,376,"{'database': 'online', 'api_gateway': 'degrade...",False,True,False
460,2023-10-11 02:00:00+00:00,93,88,343,97,2657,2169,13,147,133,0.12,1188000,83,356,"{'database': 'online', 'api_gateway': 'degrade...",False,True,False
470,2023-10-11 07:00:00+00:00,91,84,324,89,2698,1912,11,147,134,0.12,1206000,87,379,"{'database': 'online', 'api_gateway': 'degrade...",False,True,False
480,2023-10-11 12:00:00+00:00,94,90,377,92,2804,2040,11,150,126,0.13,1224000,83,357,"{'database': 'online', 'api_gateway': 'degrade...",False,True,False


In [34]:
zscores = (dataset[cols] - dataset[cols].mean()) / dataset[cols].std()
zscores.columns = [f"{c}_zscore" for c in cols]
zscores.head()

,cpu_usage_zscore,memory_usage_zscore,latency_ms_zscore,disk_usage_zscore,network_in_kbps_zscore,network_out_kbps_zscore,io_wait_zscore,thread_count_zscore,active_connections_zscore,error_rate_zscore,temperature_celsius_zscore,power_consumption_watts_zscore
0,2.633731,2.486365,2.656657,2.515746,2.731993,2.898447,2.940230,-1.169619,2.828742,2.919028,2.718148,2.516601
1,-0.298791,-0.220325,-0.254020,-0.291335,-0.441263,-0.084606,-0.371669,-0.894544,-0.331252,-0.382293,-0.647499,-0.300347
2,-0.380250,0.050344,-0.298800,-0.191082,-0.105408,-0.229967,-0.371669,-0.619469,-0.415518,-0.382293,-0.518051,-0.601186
3,-0.624627,-0.761662,-0.209240,-0.592093,-0.628879,-0.312127,-0.371669,0.343294,-0.331252,-0.382293,-0.129707,-0.655884
4,-0.706086,-0.084990,-0.492845,-0.291335,-0.638144,-0.305807,-0.371669,-0.894544,-0.457651,-0.382293,-0.518051,-0.409743


In [35]:
threshold = 3
anomaly_flags = zscores.abs() > threshold
anomaly_flags.columns = cols

def find_causes(row):
    return ", ".join(
        f"{col} (z={zscores.loc[row.name, f'{col}_zscore']:.2f})"
        for col in cols if row[col]
    )

dataset["anomaly_cause"] = anomaly_flags.apply(find_causes, axis=1)
dataset[dataset["anomaly_cause"] != ""][["timestamp", "anomaly_cause"]]

,timestamp,anomaly_cause
15,2023-10-01 19:30:00+00:00,thread_count (z=4.74)
20,2023-10-01 22:00:00+00:00,"cpu_usage (z=3.04), latency_ms (z=3.16), io_wa..."
30,2023-10-02 03:00:00+00:00,"latency_ms (z=3.18), io_wait (z=3.31), tempera..."
40,2023-10-02 08:00:00+00:00,"memory_usage (z=3.16), network_in_kbps (z=3.45..."
50,2023-10-02 13:00:00+00:00,"cpu_usage (z=3.12), memory_usage (z=3.30), net..."
60,2023-10-02 18:00:00+00:00,"cpu_usage (z=3.04), latency_ms (z=3.34), io_wa..."
75,2023-10-03 01:30:00+00:00,thread_count (z=3.09)
80,2023-10-03 04:00:00+00:00,"latency_ms (z=3.18), network_in_kbps (z=3.43),..."
90,2023-10-03 09:00:00+00:00,"memory_usage (z=3.30), latency_ms (z=3.36), te..."
100,2023-10-03 14:00:00+00:00,"latency_ms (z=3.40), disk_usage (z=3.02), netw..."


In [36]:
for i, row in dataset.iterrows():
    if row["anomaly_cause"] and row['api_gateway'] == 1:
        print(f"Anomaly detected at {row['timestamp']}: {row['anomaly_cause']} - Status: {row[['database', 'api_gateway', 'cache']].to_dict()}")

Anomaly detected at 2023-10-01 22:00:00+00:00: cpu_usage (z=3.04), latency_ms (z=3.16), io_wait (z=3.68), power_consumption_watts (z=3.04) - Status: {'database': False, 'api_gateway': True, 'cache': False}
Anomaly detected at 2023-10-02 03:00:00+00:00: latency_ms (z=3.18), io_wait (z=3.31), temperature_celsius (z=3.24) - Status: {'database': False, 'api_gateway': True, 'cache': False}
Anomaly detected at 2023-10-02 08:00:00+00:00: memory_usage (z=3.16), network_in_kbps (z=3.45), active_connections (z=3.12), power_consumption_watts (z=3.15) - Status: {'database': False, 'api_gateway': True, 'cache': False}
Anomaly detected at 2023-10-02 13:00:00+00:00: cpu_usage (z=3.12), memory_usage (z=3.30), network_out_kbps (z=3.37), active_connections (z=3.25), temperature_celsius (z=3.11) - Status: {'database': False, 'api_gateway': True, 'cache': False}
Anomaly detected at 2023-10-02 18:00:00+00:00: cpu_usage (z=3.04), latency_ms (z=3.34), io_wait (z=3.31), error_rate (z=3.25), temperature_celsiu

In [37]:
NUMERIC_COLS = ['cpu_usage','memory_usage','latency_ms','disk_usage','network_in_kbps','network_out_kbps','io_wait','thread_count','active_connections','error_rate','temperature_celsius','power_consumption_watts']

In [38]:
def compute_insights(df):
    insight = {
        "average_latency_ms": round(float(df["latency_ms"].mean()), 2),
        "max_cpu_usage": int(df["cpu_usage"].max()),
        "max_memory_usage": int(df["memory_usage"].max()),
        "error_rate": round(float(df["error_rate"].mean()), 4),
        "uptime_seconds": int(df["uptime_seconds"].max()),
    }
    return insight

In [39]:
compute_insights(dataset)

{'average_latency_ms': 156.02,
 'max_cpu_usage': 99,
 'max_memory_usage': 92,
 'error_rate': 0.0316,
 'uptime_seconds': 1258200}

In [40]:
def severity_from_zscore(z: float):
    z = abs(z)
    return "high" if z >= 3 else "medium" if z >= 2.5 else "low"

def detect_anomalies(df, threshold: float = 3.0):
    """Liste toutes les anomalies du dataset (une entrée par relevé x métrique
    dépassant le seuil), et non plus une seule entrée par colonne."""
    mean = df[NUMERIC_COLS].mean()
    std = df[NUMERIC_COLS].std()
    zscores = (df[NUMERIC_COLS] - mean) / std

    anomalies = []
    for idx, row_z in zscores.iterrows():
        for col in NUMERIC_COLS:
            z = row_z[col]
            if abs(z) <= threshold:
                continue
            # seuil du côté du dépassement (haut si z > 0, bas sinon)
            bound = mean[col] + (threshold if z > 0 else -threshold) * std[col]
            value = float(df.loc[idx, col])
            anomalies.append({
                "timestamp": df.loc[idx, "timestamp"],
                "metric": col,
                "value": value,
                "threshold": round(float(bound), 2),
                "zscore": round(float(z), 2),
                "severity": severity_from_zscore(z),
                "description": f"{col} a atteint {value} (z={z:.2f}), au-delà du seuil normal ({bound:.2f}).",
            })

    # par relevé chronologique, puis de l'écart le plus fort au plus faible
    anomalies.sort(key=lambda a: (a["timestamp"], -abs(a["zscore"])))
    return anomalies

In [41]:
zscores

,cpu_usage_zscore,memory_usage_zscore,latency_ms_zscore,disk_usage_zscore,network_in_kbps_zscore,network_out_kbps_zscore,io_wait_zscore,thread_count_zscore,active_connections_zscore,error_rate_zscore,temperature_celsius_zscore,power_consumption_watts_zscore
0,2.633731,2.486365,2.656657,2.515746,2.731993,2.898447,2.940230,-1.169619,2.828742,2.919028,2.718148,2.516601
1,-0.298791,-0.220325,-0.254020,-0.291335,-0.441263,-0.084606,-0.371669,-0.894544,-0.331252,-0.382293,-0.647499,-0.300347
2,-0.380250,0.050344,-0.298800,-0.191082,-0.105408,-0.229967,-0.371669,-0.619469,-0.415518,-0.382293,-0.518051,-0.601186
3,-0.624627,-0.761662,-0.209240,-0.592093,-0.628879,-0.312127,-0.371669,0.343294,-0.331252,-0.382293,-0.129707,-0.655884
4,-0.706086,-0.084990,-0.492845,-0.291335,-0.638144,-0.305807,-0.371669,-0.894544,-0.457651,-0.382293,-0.518051,-0.409743
...,...,...,...,...,...,...,...,...,...,...,...,...
495,0.923093,-0.084990,0.507234,0.310182,0.072943,0.876038,-0.003680,2.956506,0.047948,0.608103,0.129189,0.520123
496,-0.461709,-0.220325,-0.358506,-0.391588,-0.557076,-0.375327,-0.371669,0.480831,-0.289118,-0.382293,-0.518051,-0.245649
497,-0.298791,-0.626328,-0.313726,-0.090829,-0.207323,-0.381647,-0.739657,-0.757006,-0.289118,-0.382293,-0.129707,-0.464441
498,-0.298791,-0.761662,-0.522698,-0.592093,-0.369460,-0.691329,-0.371669,-1.032082,-0.331252,-0.382293,-0.259155,-0.464441


In [42]:
anomalies = detect_anomalies(dataset)
print(f"{len(anomalies)} anomalies détectées sur {len(dataset)} relevés")
test = pd.DataFrame(anomalies).groupby("timestamp").agg(list)
test

197 anomalies détectées sur 500 relevés


,metric,value,threshold,zscore,severity,description
timestamp,,,,,,
2023-10-01 19:30:00+00:00,[thread_count],[186.0],[173.32],[4.74],[high],"[thread_count a atteint 186.0 (z=4.74), au-del..."
2023-10-01 22:00:00+00:00,"[io_wait, latency_ms, cpu_usage, power_consump...","[14.0, 368.0, 98.0, 375.0]","[12.16, 357.0, 97.5, 373.68]","[3.68, 3.16, 3.04, 3.04]","[high, high, high, high]","[io_wait a atteint 14.0 (z=3.68), au-delà du s..."
2023-10-02 03:00:00+00:00,"[io_wait, temperature_celsius, latency_ms]","[13.0, 88.0, 369.0]","[12.16, 86.18, 357.0]","[3.31, 3.24, 3.18]","[high, high, high]","[io_wait a atteint 13.0 (z=3.31), au-delà du s..."
2023-10-02 08:00:00+00:00,"[network_in_kbps, memory_usage, power_consumpt...","[2850.0, 91.0, 379.0, 133.0]","[2656.71, 89.8, 373.68, 130.06]","[3.45, 3.16, 3.15, 3.12]","[high, high, high, high]","[network_in_kbps a atteint 2850.0 (z=3.45), au..."
2023-10-02 13:00:00+00:00,"[network_out_kbps, memory_usage, active_connec...","[2287.0, 92.0, 136.0, 99.0, 87.0]","[2169.14, 89.8, 130.06, 97.5, 86.18]","[3.37, 3.3, 3.25, 3.12, 3.11]","[high, high, high, high, high]","[network_out_kbps a atteint 2287.0 (z=3.37), a..."
2023-10-02 18:00:00+00:00,"[latency_ms, io_wait, error_rate, temperature_...","[380.0, 13.0, 0.13, 88.0, 382.0, 98.0]","[357.0, 12.16, 0.12, 86.18, 373.68, 97.5]","[3.34, 3.31, 3.25, 3.24, 3.23, 3.04]","[high, high, high, high, high, high]","[latency_ms a atteint 380.0 (z=3.34), au-delà ..."
2023-10-03 01:30:00+00:00,[thread_count],[174.0],[173.32],[3.09],[high],"[thread_count a atteint 174.0 (z=3.09), au-del..."
2023-10-03 04:00:00+00:00,"[network_in_kbps, network_out_kbps, active_con...","[2843.0, 2300.0, 135.0, 369.0]","[2656.71, 2169.14, 130.06, 357.0]","[3.43, 3.41, 3.21, 3.18]","[high, high, high, high]","[network_in_kbps a atteint 2843.0 (z=3.43), au..."
2023-10-03 09:00:00+00:00,"[latency_ms, memory_usage, temperature_celsius]","[381.0, 92.0, 88.0]","[357.0, 89.8, 86.18]","[3.36, 3.3, 3.24]","[high, high, high]","[latency_ms a atteint 381.0 (z=3.36), au-delà ..."


In [43]:
SEVERITY_ORDER = {"low": 0, "medium": 1, "high": 2}

def group_anomalies_by_timestamp(anomalies: list[dict]):
    """Regroupe les anomalies par relevé : un événement = un timestamp avec
    toutes ses métriques anormales, triées par écart décroissant."""
    events = {}
    for a in anomalies:
        event = events.setdefault(a["timestamp"], {"timestamp": a["timestamp"], "metrics": []})
        event["metrics"].append({k: a[k] for k in ("metric", "value", "threshold", "zscore", "severity")})

    for event in events.values():
        event["metrics"].sort(key=lambda m: -abs(m["zscore"]))
        event["n_metrics"] = len(event["metrics"])
        event["severity"] = max((m["severity"] for m in event["metrics"]), key=SEVERITY_ORDER.get)
        event["max_zscore"] = event["metrics"][0]["zscore"]

    return sorted(events.values(), key=lambda e: e["timestamp"])

events = group_anomalies_by_timestamp(anomalies)
print(f"{len(events)} relevés anormaux, "
      f"{sum(e['severity'] == 'high' for e in events)} de sévérité high")
events[0]

59 relevés anormaux, 59 de sévérité high


{'timestamp': Timestamp('2023-10-01 19:30:00+0000', tz='UTC'),
 'metrics': [{'metric': 'thread_count',
   'value': 186.0,
   'threshold': 173.32,
   'zscore': 4.74,
   'severity': 'high'}],
 'n_metrics': 1,
 'severity': 'high',
 'max_zscore': 4.74}

In [44]:
events[2]

{'timestamp': Timestamp('2023-10-02 03:00:00+0000', tz='UTC'),
 'metrics': [{'metric': 'io_wait',
   'value': 13.0,
   'threshold': 12.16,
   'zscore': 3.31,
   'severity': 'high'},
  {'metric': 'temperature_celsius',
   'value': 88.0,
   'threshold': 86.18,
   'zscore': 3.24,
   'severity': 'high'},
  {'metric': 'latency_ms',
   'value': 369.0,
   'threshold': 357.0,
   'zscore': 3.18,
   'severity': 'high'}],
 'n_metrics': 3,
 'severity': 'high',
 'max_zscore': 3.31}

In [45]:
STATUS_RANK = {"online": 0, "degraded": 1, "offline": 2}
STATUS_SEVERITY = {"online": "low", "degraded": "medium", "offline": "high"}
STATE_COLS = [f"{s}_raw" for s in SERVICES]

def worst(severities):
    return max(severities, key=SEVERITY_ORDER.get)

def compute_baseline(dataset_states):
    """Baseline = les relevés où les trois services sont `online`.

    Rend (deltas, stats) :
    - deltas : écart en % de la moyenne de chaque groupe d'état à cette baseline.
      Second signal, complémentaire du z-score : le seuil z > 3 est calculé sur la
      distribution globale et ne voit pas une dérive de fond propre à un groupe
      (ex. api_gateway+cache degraded : +68 % de latence, aucun dépassement z > 3) ;
    - stats : plages saines par métrique, pour que les paramètres proposés
      (plafonds, seuils) restent compatibles avec le trafic normal.
    """
    all_online = (dataset_states[STATE_COLS] == "online").all(axis=1)
    if not all_online.any():
        return pd.DataFrame(columns=NUMERIC_COLS), pd.DataFrame()
    healthy = dataset_states.loc[all_online, NUMERIC_COLS]
    group_mean = dataset_states.groupby(STATE_COLS)[NUMERIC_COLS].mean()
    deltas = ((group_mean / healthy.mean().replace(0, pd.NA) - 1) * 100).round(0)
    stats = healthy.agg(["mean", "min", "max"]).T.join(
        healthy.quantile(0.95).rename("p95")
    ).round(2)
    return deltas, stats

def format_baseline_ranges(stats):
    return "\n".join(
        f"  - {metric}: moyenne {row['mean']}, p95 {row['p95']}, "
        f"observé {row['min']}→{row['max']}"
        for metric, row in stats.iterrows()
    )

def summarize_by_service_state(dataset, raw_status_df, anomalies):
    """Regroupe les relevés par état des services (database, api_gateway, cache).

    Rend (state_summary, metric_profile, baseline_deltas) :
    - state_summary : un état par ligne — relevés observés, relevés anormaux, sévérité ;
    - metric_profile : (état x métrique) — récurrence et amplitude des dépassements z > 3 ;
    - baseline_deltas : (état x métrique) — écart de la moyenne du groupe au tout-online.

    `dataset[SERVICES]` ne contient que le booléen : on rattache l'état brut
    (online/degraded/offline) depuis raw_status_df."""
    dataset_states = dataset.join(raw_status_df[SERVICES].add_suffix("_raw"))
    anomalies_df = pd.DataFrame(anomalies).merge(
        dataset_states[["timestamp", *STATE_COLS]], on="timestamp"
    )

    metric_profile = (
        anomalies_df.groupby([*STATE_COLS, "metric"])
        .agg(n_events=("timestamp", "nunique"), threshold=("threshold", "first"),
             value_min=("value", "min"), value_max=("value", "max"),
             mean_abs_zscore=("zscore", lambda s: round(s.abs().mean(), 2)),
             max_abs_zscore=("zscore", lambda s: round(s.abs().max(), 2)))
        .sort_values(["n_events", "max_abs_zscore"], ascending=False)
    )

    state_summary = (
        dataset_states.groupby(STATE_COLS).size().rename("n_samples").to_frame()
        .join(anomalies_df.groupby(STATE_COLS).agg(
            n_events=("timestamp", "nunique"),
            timestamps=("timestamp", lambda s: sorted(s.unique())),
            anomaly_severity=("severity", worst)))
    )
    state_summary["n_events"] = state_summary["n_events"].fillna(0).astype(int)
    state_summary["severity"] = [
        worst([STATUS_SEVERITY[max(state, key=STATUS_RANK.get)],
               severity if isinstance(severity, str) else "low"])
        for state, severity in zip(state_summary.index, state_summary["anomaly_severity"])
    ]

    # états à traiter : un service non-online, ou des relevés anormaux. Un service
    # `offline` sans métrique hors seuil reste un incident (cas database=offline).
    degraded = pd.Series([any(v != "online" for v in state) for state in state_summary.index],
                         index=state_summary.index)
    state_summary = (
        state_summary[degraded | (state_summary["n_events"] > 0)]
        .drop(columns="anomaly_severity")
        .sort_values(["severity", "n_events"],
                     key=lambda c: c.map(SEVERITY_ORDER) if c.name == "severity" else c,
                     ascending=False)
    )
    baseline_deltas, baseline_stats = compute_baseline(dataset_states)
    return state_summary, metric_profile, baseline_deltas, baseline_stats

def build_clusters(state_summary, metric_profile, baseline_deltas, baseline_stats):
    """Un groupe = une combinaison d'états des services, prête pour le prompt :
    c'est l'unité de diagnostic, les mêmes services dégradés ont la même cause racine."""
    profiles = {state: df.reset_index(level=STATE_COLS, drop=True).reset_index().to_dict("records")
                for state, df in metric_profile.groupby(level=STATE_COLS)}
    return [{
        "id": "|".join(f"{s}={v}" for s, v in zip(SERVICES, state)),
        "service_state": dict(zip(SERVICES, state)),
        "n_samples": int(row["n_samples"]),
        "n_events": int(row["n_events"]),
        "severity": row["severity"],
        "timestamps": row["timestamps"] if isinstance(row["timestamps"], list) else [],
        "metrics": profiles.get(state, []),
        "baseline_delta": (baseline_deltas.loc[state].dropna().to_dict()
                           if state in baseline_deltas.index else {}),
        "baseline_ranges": format_baseline_ranges(baseline_stats),
    } for state, row in state_summary.iterrows()]

state_summary, metric_profile, baseline_deltas, baseline_stats = summarize_by_service_state(
    dataset, raw_status_df, anomalies
)
clusters = build_clusters(state_summary, metric_profile, baseline_deltas, baseline_stats)
print(f"{len(events)} relevés anormaux → {len(clusters)} groupes d'état des services")
baseline_deltas.loc[state_summary.index].T

59 relevés anormaux → 4 groupes d'état des services


database_raw              online          offline   online
api_gateway_raw         degraded   online  online degraded
cache_raw                 online degraded  online degraded
cpu_usage                   72.0     27.0     9.0     35.0
memory_usage                36.0      7.0    -0.0     16.0
latency_ms                 167.0     56.0    -4.0     68.0
disk_usage                  53.0     16.0     0.0     26.0
network_in_kbps            117.0     21.0    -2.0     35.0
network_out_kbps            93.0     30.0    -1.0     35.0
io_wait                    296.0     67.0    -2.0     92.0
thread_count                -0.0     20.0    -0.0      7.0
active_connections         159.0     20.0     1.0     42.0
error_rate                 505.0    150.0     0.0    100.0
temperature_celsius         41.0      8.0    -0.0     17.0
power_consumption_watts     48.0      8.0    -1.0     13.0

# Generateur de recommandations

In [46]:
class Recommendation(BaseModel):
    id: str
    action: str
    target: str
    parameters: dict
    benefit_estimate: str

class RecommendationList(BaseModel):
    recommendations: list[Recommendation]

In [47]:
llm = ChatOpenAI(model="gpt-5.4", temperature=0)

# ---------- 1 état des services: 1 recommandation ----------
cluster_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Tu es SRE. On te donne tous les relevés qui partagent le MÊME état des services "
     "(database / api_gateway / cache). Tu proposes UNE action corrective qui traite la cause "
     "racine commune à ce groupe, jamais une action par métrique. `target` est un service "
     "existant (database, api_gateway ou cache), `parameters` contient les valeurs concrètes "
     "de l'action (seuils, tailles, durées) — cohérentes avec les plages normales indiquées.\n"
     "Deux signaux distincts te sont donnés :\n"
     "- les dépassements z > 3, qui sont des pics ponctuels ;\n"
     "- l'écart des moyennes du groupe à la baseline tout-online, qui est la dérive de fond "
     "du groupe et existe même sans aucun pic.\n"
     "Un groupe sans pic mais dont les moyennes dérivent reste une dégradation réelle. "
     "Un groupe plat sur les deux signaux, avec un service `offline` ou `degraded`, signale "
     "une panne applicative et non une saturation de ressource."),
    ("human",
     "État des services : {service_state}\n"
     "{n_events} relevés anormaux sur {n_samples} dans cet état — sévérité {severity}\n"
     "Relevés anormaux (timestamps) :\n{period}\n\n"
     "Pics (dépassements z > 3, récurrence sur le groupe) :\n{metrics}\n\n"
     "Dérive de fond (moyenne du groupe vs baseline tout-online) :\n{baseline_delta}\n\n"
     "Plages observées hors dégradation (baseline tout-online) :\n{baseline_ranges}\n\n"
     "Utilise `{cluster_id}` comme `id`."),
])

def cluster_to_vars(c: dict):
    deltas = sorted(c["baseline_delta"].items(), key=lambda kv: -abs(kv[1]))
    return {
        "cluster_id": c["id"],
        "service_state": ", ".join(f"{s}: {v}" for s, v in c["service_state"].items()),
        "n_events": c["n_events"],
        "n_samples": c["n_samples"],
        "severity": c["severity"],
        "period": "\n".join(f"  - {t:%Y-%m-%d %H:%M}" for t in c["timestamps"])
                  or "  aucun relevé anormal",
        "metrics": "\n".join(
            f"  - {m['metric']}: {m['n_events']}/{c['n_events']} relevés, "
            f"|z| moyen {m['mean_abs_zscore']} (max {m['max_abs_zscore']}), "
            f"valeurs {m['value_min']}→{m['value_max']} (seuil {m['threshold']})"
            for m in c["metrics"]
        ) or "  Aucun pic détecté : aucune métrique ne dépasse z > 3.",
        "baseline_delta": "\n".join(
            f"  - {metric}: {delta:+.0f}%" for metric, delta in deltas if abs(delta) >= 5
        ) or "  Aucune dérive : toutes les métriques à moins de 5 % de la baseline.",
        "baseline_ranges": c["baseline_ranges"],
    }

cluster_chain = (
    RunnableLambda(cluster_to_vars)
    | cluster_prompt
    | llm.with_structured_output(Recommendation, method="function_calling")
).with_retry(stop_after_attempt=3)

def generate_recommendations(clusters: list[dict]):
    """Une recommandation par état des services : 4 appels ici, contre 59 avec un appel
    par relevé anormal. Les groupes sont disjoints par construction — un état des
    services = une cause racine —, il n'y a donc pas de doublons à consolider."""
    if not clusters:
        return []
    return cluster_chain.batch(clusters, config={"max_concurrency": 4})

In [48]:
def compute_service_status_summary(raw_status_df):
    """Classe chaque service selon l'état le plus dégradé observé sur la période."""
    summary = {"online": [], "degraded": [], "offline": []}
    for service in SERVICES:
        worst_status = max(raw_status_df[service], key=STATUS_RANK.get)
        summary[worst_status].append(service)
    return summary

def format_report_anomaly(a: dict):
    """Anomalie au format de sortie attendu ; le relevé concerné passe dans la description."""
    return {
        "metric": a["metric"],
        "value": a["value"],
        "threshold": a["threshold"],
        "severity": a["severity"],
        "description": f"[{a['timestamp']:%Y-%m-%d %H:%M}] {a['description']}",
    }

def make_report(dataset, raw_status_df):
    anomalies = detect_anomalies(dataset)
    # recommandations générées par état des services, pas par relevé
    summary, profile, deltas, stats = summarize_by_service_state(dataset, raw_status_df, anomalies)
    recommendations = generate_recommendations(build_clusters(summary, profile, deltas, stats))

    report = {
        "timestamp": pd.Timestamp.now(tz="UTC").isoformat(),
        "insights": compute_insights(dataset),
        "anomalies": [format_report_anomaly(a) for a in anomalies],
        "recommendations": [rec.model_dump() for rec in recommendations],
        "service_status_summary": compute_service_status_summary(raw_status_df),
    }
    return json.loads(json.dumps(report, default=str))

In [49]:
for i, row in pd.DataFrame(events).iterrows():
    if row["severity"] == "high":
        print(row["metrics"])

[{'metric': 'thread_count', 'value': 186.0, 'threshold': 173.32, 'zscore': 4.74, 'severity': 'high'}]
[{'metric': 'io_wait', 'value': 14.0, 'threshold': 12.16, 'zscore': 3.68, 'severity': 'high'}, {'metric': 'latency_ms', 'value': 368.0, 'threshold': 357.0, 'zscore': 3.16, 'severity': 'high'}, {'metric': 'cpu_usage', 'value': 98.0, 'threshold': 97.5, 'zscore': 3.04, 'severity': 'high'}, {'metric': 'power_consumption_watts', 'value': 375.0, 'threshold': 373.68, 'zscore': 3.04, 'severity': 'high'}]
[{'metric': 'io_wait', 'value': 13.0, 'threshold': 12.16, 'zscore': 3.31, 'severity': 'high'}, {'metric': 'temperature_celsius', 'value': 88.0, 'threshold': 86.18, 'zscore': 3.24, 'severity': 'high'}, {'metric': 'latency_ms', 'value': 369.0, 'threshold': 357.0, 'zscore': 3.18, 'severity': 'high'}]
[{'metric': 'network_in_kbps', 'value': 2850.0, 'threshold': 2656.71, 'zscore': 3.45, 'severity': 'high'}, {'metric': 'memory_usage', 'value': 91.0, 'threshold': 89.8, 'zscore': 3.16, 'severity': 'hi

## Format de sortie attendu

```json
{
    "timestamp": "string (ISO 8601)",
    "insights": {
        "average_latency_ms": "number",
        "max_cpu_usage": "number",
        "max_memory_usage": "number",
        "error_rate": "number",
        "uptime_seconds": "number"
    },
    "anomalies": [{
        "metric": "string",
        "value": "number",
        "threshold": "number",
        "severity": "string (low|medium|high)",
        "description": "string"
    }],
    "recommendations": [{
        "id": "string",
        "action": "string",
        "target": "string",
        "parameters": "object",
        "benefit_estimate": "string"
    }],
    "service_status_summary": {
        "online": ["string"],
        "degraded": ["string"],
        "offline": ["string"]
    }
}
```


In [50]:
report = make_report(dataset, raw_status_df)

In [51]:
report["recommendations"]

[{'id': 'database=online|api_gateway=degraded|cache=online',
  'action': 'scale_out',
  'target': 'api_gateway',
  'parameters': {'instance_count': 3,
   'cpu_target_percent': 60,
   'memory_target_percent': 70,
   'connection_pool_limit': 140},
  'benefit_estimate': 'Réduit la saturation structurelle de l’api_gateway responsable de la dérive de latence/erreurs et absorbe les pics récurrents de trafic, avec baisse attendue de la latence et du taux d’erreur vers la baseline.'},
 {'id': 'database=online|api_gateway=online|cache=degraded',
  'action': 'Augmenter la capacité et la concurrence du cache pour résorber la saturation récurrente des threads',
  'target': 'cache',
  'parameters': {'min_replicas': 2,
   'max_replicas': 4,
   'worker_threads': 192,
   'active_connection_limit': 64,
   'cpu_target_percent': 65,
   'scale_up_cooldown_minutes': 10},
  'benefit_estimate': 'Réduction attendue de la latence de 30–45%, baisse du taux d’erreur de 50–70% et retour du thread_count sous ~157 

In [52]:
with open("output.json", "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)